In [ ]:
from typing import Iterable, Iterator, Callable
from itertools import chain as iterchain, combinations as itercomb
from collections import Counter
from pprint import pprint

iterflat = iterchain.from_iterable

In [ ]:
from board import DIGITS, Digits, Loc, Cell, Board

In [ ]:
def boardiff(b0: Board, b2: Board) -> Iterable[Cell]:
    """Compare boards, yield removed content"""
    for c0, c2 in zip(iter(b0), iter(b2)):
        if c0.digits != c2.digits:
            yield Cell(c0.loc, Digits(c0.digits - c2.digits))


# Topology

The board is classicaly 9 rows, 9 columns, and 9 boxes over them. These are major units.

Intersection of a row and a column gives individual cell. A cell is subdivided further into 9 segments for digits, but only for visualizing purposes.

Intersection of a box with a row or column gives sector.

Generalized, all the localities can be addressed by a set of `(box, col, row)`

Visibility of 2 localities means they share some unit (or two). It determines location of cell peers and possibly conflicting drafts.


In [ ]:
from topology import Zone, peers, peerz, allpeers, visibility, allvisible

In [ ]:
# construction
assert Zone.B(1) == Zone(1, ..., ...)
assert Zone.R(2) == Zone(..., 2, ...)
assert Zone.C(3) == Zone(..., ..., 3)
assert Zone.L(Loc(2, 3)) == Zone(1, 2, 3)  # fulfiled box

assert Zone(1, 2, 3).is_cell
assert Zone(1, ..., ...).is_unit
assert Zone(1, 2, ...).is_sector

# intersection
assert Zone.R(2) & Zone.C(3) == Zone(1, 2, 3)
assert Zone.B(1) & Zone.R(2) == Zone(1, 2, ...)
assert Zone.B(1) & Zone.C(3) == Zone(1, ..., 3)

# containering
assert set(iter(Zone(1, ..., ...))) == {
    Loc(1, 1),
    Loc(1, 2),
    Loc(1, 3),
    Loc(2, 1),
    Loc(2, 2),
    Loc(2, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}
assert set(iter(Zone(1, 2, ...))) == {Loc(2, 1), Loc(2, 2), Loc(2, 3)}
assert Loc(2, 3) in Zone.B(1)
assert Loc(2, 3) in Zone.R(2)
assert Loc(2, 3) in Zone.C(3)

In [ ]:
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(2, 3))) == {Zone.B(1)}
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(1, 3))) == {Zone.R(1), Zone.B(1)}
assert visibility(Zone(1, 2, ...), Zone.L(Loc(2, 9))) == {Zone.R(2)}
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(8, 9))) == set()

assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(8, 9))) == {Zone(3, 1, 9), Zone(7, 8, 2)}
assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(2, 3))) == {Zone(1, ..., ...)}
assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(3, 2))) == {Zone(1, ..., ...), Zone(..., ..., 2)}

In [ ]:
print(*map(str, Zone.Units()))

In [ ]:
assert peers(Loc(1, 2), Zone.B(1)) == {
    Loc(1, 1),
    Loc(1, 3),
    Loc(2, 1),
    Loc(2, 2),
    Loc(2, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}
assert peerz(Zone(1, 2, ...), Zone.B(1)) == {
    Loc(1, 1),
    Loc(1, 2),
    Loc(1, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}

# Targeting

Generic draft target:

- location: cell, unit, sector
- digits: one or more digits


In [ ]:
from analysis import Node, cellmatching, cellspoiling

In [ ]:
assert Node.C(Cell(Loc(1, 2), Digits({1, 2, 3}))) == Node(Zone.L(Loc(1, 2)), Digits({1, 2, 3}))
assert Node.at(Loc(1, 2), Digits({1, 2, 3})) == Node(Zone.L(Loc(1, 2)), Digits({1, 2, 3}))
assert Node.at(Loc(1, 2), 5) == Node(Zone.L(Loc(1, 2)), Digits({5}))

assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_cellular
assert Node(Zone.L(Loc(1, 2)), Digits({1})).loc == Loc(1, 2)
assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_singular
assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_casual
assert not Node(Zone.L(Loc(1, 2)), Digits({1, 2})).is_singular
assert not Node(Zone.L(Loc(1, 2)), Digits({1, 2})).is_casual
assert not Node(Zone(1, 2, ...), Digits({1})).is_cellular
assert not Node(Zone(1, 2, ...), Digits({1})).is_casual

# Resolving

searching for patterns -> generating resolutions -> applying resolutions


In [ ]:
from utils import filt_finals, filt_having, filt_havesome, draftborhood, draftboard, flat_cells, count_finals, count_digits, grab_digits
from solving import Resolver, Resolving, Pattern, solver, onceolver
from searching import search_breadth
from analysis import Link, HLink, SLink, Chain

In [ ]:
async def solve_silent(initial: Board, *resolvers: Resolver):
    result = initial
    _, _, drafted = result.validate()
    assert drafted
    async for _, _, result in solver(initial, *resolvers):
        complete, valid, drafted = result.validate()
        if complete or not valid or not drafted:
            break
    return result

In [ ]:
async def solve_logging(initial: Board, /, *resolvers: Resolver, filtout: set[str] | None = None):
    result = initial
    iterations = 0
    _, _, drafted = result.validate()
    assert drafted
    current = result
    async for resolver, pattern, result in solver(initial, *resolvers):
        iterations += 1
        if filtout and resolver.__name__ in filtout:
            continue
        print(f"{iterations:03d} {resolver.__name__}", end=": ")
        diff = list(boardiff(current, result))
        print("-", " ".join(map(str, diff)))
        # pprint(rule)

        current = result
        complete, valid, drafted = result.validate()
        if complete or not valid or not drafted:
            break

    complete, valid, drafted = result.validate()
    stuck = not complete and not drafted
    print("========")
    print(f"{iterations=} {complete=} {valid=} {stuck=}")
    return result


# Resolvers/Rules


## Random choice


In [ ]:
def random_choice(board: Board) -> Resolving:
    drafts = list(draftboard(board))
    assert len(drafts)

    drafts.sort(key=lambda c: len(c))
    leastcell = drafts[0]
    counts = count_digits(drafts)

    digits = list(leastcell.digits)
    digits.sort(key=lambda d: counts[d])
    leastdig = digits[0]

    chosen = Node.at(leastcell, leastdig)
    loosen = Node.at(leastcell, leastcell.digits - chosen.digits)

    yield Pattern(
        anchors={chosen},
        spoilers={loosen},
    )

## Rectangles

TODO


## Singles


In [ ]:
def naked_singles(board: Board) -> Resolving:
    for fincell in filter(filt_finals, iter(board)):
        findig = Digits({fincell.final})
        anchor = Node.at(fincell, findig)
        spoilers = set(filter(filt_havesome(findig), board.slice(allpeers(anchor.zone))))
        if spoilers:
            yield Pattern(
                anchors={anchor},
                spoilers={Node.at(c.loc, anchor.digits) for c in spoilers},
            )

In [ ]:
def hidden_singles(board: Board) -> Resolving:
    for unit in Zone.Units():
        drafts = set(draftborhood(board, unit))
        counts = count_digits(drafts)
        for dig, cnt in counts.items():
            if cnt == 1:
                [cell] = filter(lambda c: dig in c, drafts)
                anchor = Node.at(cell, dig)
                spoilers = Node.at(cell, cell.digits - {dig})
                yield Pattern(
                    space={Node.at(unit, dig)},
                    anchors={anchor},
                    spoilers={spoilers},
                )


## Multiples


In [ ]:
def naked_multiples(board: Board) -> Resolving:
    for unit in Zone.Units():
        drafts = tuple(draftborhood(board, unit))
        digits = grab_digits(drafts)
        for m in range(2, 5):
            for combo in itercomb(digits, m):
                combits = Digits(combo)
                habitat = set(filter(lambda c: c.digits & combits, drafts))
                naked = set(filter(lambda c: c.digits <= combits, habitat))
                # print(unit, combits, set(map(str, naked)), "/", set(map(str, habitat)))
                if len(naked) == len(combo):
                    spoilers = set(filter(filt_havesome(combits), drafts)) - naked
                    if spoilers:
                        yield Pattern(
                            space={Node.at(unit, combits)},
                            anchors={Node.at(c, combits) for c in naked},
                            spoilers={Node.at(c, combits) for c in spoilers},
                        )

In [ ]:
def hidden_multiples(board: Board) -> Resolving:
    for unit in Zone.Units():
        drafts = tuple(draftborhood(board, unit))
        digits = grab_digits(drafts)
        for m in range(2, 5):
            for combo in itercomb(digits, m):
                combits = Digits(combo)
                habitat = set(filter(lambda c: c.digits & combits, drafts))
                if len(habitat) == len(combo):
                    spoilers = set(filter(lambda c: c.digits > combits, habitat))
                    if spoilers:
                        yield Pattern(
                            space={Node.at(unit, combits)},
                            anchors={Node.at(c, c.digits & combits) for c in habitat},
                            spoilers={Node.at(c, c.digits - combits) for c in spoilers},
                        )

## Triplets


In [ ]:
def iter_sect() -> Iterator[tuple[Zone, Zone, Zone]]:
    for box in Zone.Allbox():
        for side in Zone.across(box):
            yield box, side, box & side  # type: ignore impossible null

In [ ]:
def locked_triplets(board: Board) -> Resolving:
    for box, side, sect in iter_sect():
        boxcounts = count_digits(draftborhood(board, box))
        sidecounts = count_digits(draftborhood(board, side))
        sectcounts = count_digits(draftborhood(board, sect))
        # print(box, side, sect, sectcounts)
        for dig, cnt in sectcounts.items():
            if cnt < 2:
                continue
            elif cnt == boxcounts[dig] and cnt < sidecounts[dig]:
                sideborhood = set(iter(side)) - set(iter(box))
                yield Pattern(
                    anchors={Node.at(sect, dig)},
                    space={Node.at(box, dig)},
                    spoilers={Node.at(z, dig) for z in sideborhood},
                )
            elif cnt == sidecounts[dig] and cnt < boxcounts[dig]:
                insideborhood = set(iter(box)) - set(iter(side))
                yield Pattern(
                    anchors={Node.at(sect, dig)},
                    space={Node.at(side, dig)},
                    spoilers={Node.at(z, dig) for z in insideborhood},
                )


## Links/Chains


### Strong/Hard links

Criteria:

- (bi-location) only 2 drafts of same digit in a unit
- (bi-value) only 2 drafts in a cell


In [ ]:
def search_hard_1(board: Board) -> Iterable[HLink]:
    """search for biloc single-value links"""
    for zone in Zone.Units():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            if cnt == 2:
                (n1, n2) = filter(filt_having(dig), drafts)
                yield HLink((Node.at(n1, dig), Node.at(n2, dig)))

In [ ]:
def search_hard_2(board: Board):
    """search for bivalue links"""
    for cell in filter(lambda c: len(c) == 2, draftboard(board)):
        d1, d2 = cell.digits
        yield HLink((Node.at(cell, d1), Node.at(cell, d2)))

- any 2 triplets/singles that partition all habitants in a unit


In [ ]:
def search_hard_31(board: Board):
    """search for triplets and singles"""
    for unit in Zone.Units():
        drafts = set(draftborhood(board, unit))

        def subcount(dig: int):
            for sect in Zone.sectors(unit):
                subhab = tuple(filter(lambda c: dig in c and c.loc in sect, drafts))
                habcnt = len(subhab)
                if habcnt > 1:
                    yield Node.at(sect, dig), habcnt
                elif habcnt == 1:
                    yield Node.at(subhab[0], dig), habcnt

        habitants = count_digits(drafts)
        # print(unit, habitants)
        for dig, cnt in habitants.items():
            subcounts = {n: cnt for n, cnt in subcount(dig)}  # collapsing overlapping singular nodes
            for h1, h2 in itercomb(subcounts.keys(), 2):
                print(
                    h1,
                    h2,
                )
                if not (h1.zone & h2.zone) and subcounts[h1] + subcounts[h2] == cnt:  # full partition
                    yield HLink((h1, h2))

### Weak/Soft links

Criteria:

- (bi-val) any 2 draft in a cell
- (bi-loc) any 2 drafts of same digit in a unit
- (bi-tripl) any 2 triplets in a unit == hard link
- (triploc) triplet and any singular draft of the same digit within shared unit


In [ ]:
def check_soft_2(n1: Node, n2: Node):
    return n1.dig != n2.dig and n1.is_cellular and n2.is_cellular and n1.loc == n2.loc


def connect_soft_2(n1: Node, n2: Node) -> SLink | None:
    """bi-value"""
    assert n1 != n2

    if check_soft_2(n1, n2):
        return SLink((n1, n2))

In [ ]:
def check_soft_13(n1: Node, n2: Node):
    return n1.dig == n2.dig and not (n1.zone & n2.zone) and visibility(n1.zone, n2.zone)


def connect_soft_13(n1: Node, n2: Node) -> SLink | None:
    """inter-location, both singulars and triplets"""
    assert n1 != n2

    if check_soft_13(n1, n2):
        return SLink((n1, n2))

In [ ]:
Connecting = Callable[[Node, Node], SLink | None]

### Chains

Alternating inference chains, with simple bilocation links

X-Wing, X-Cycle, Nice Loop, etc


In [ ]:
def scan_visible_2(board: Board, n1: Node, n2: Node) -> Iterable[Node]:
    if not check_soft_2(n1, n2):
        return

    cell = board.get(n1.loc)
    anchors = n1.digits | n2.digits
    for d in filter(lambda d: d not in anchors, cell.digits):
        yield Node.at(cell, d)


In [ ]:
def scan_visible_13(board: Board, n1: Node, n2: Node) -> Iterable[Node]:
    if not all(n.is_casual for n in (n1, n2)):
        return
    anchors = n1.digits | n2.digits  # max=2
    for vizone in allvisible(n1.zone, n2.zone):  # max=2
        for cell in filter(lambda c: c.digits & anchors and c.loc not in n1.zone and c.loc not in n2.zone, draftborhood(board, vizone)):  # max=18
            for dig in anchors:  # max=36
                node = Node.at(cell, dig)
                if check_soft_13(node, n1) and check_soft_13(node, n2):
                    yield node

In [ ]:
Scanning = Callable[[Board, Node, Node], Iterable[Node]]

#### open

Pattern: a unclosed alternating hard-ended chain with matching edges

Rule: invalidate all visible from both edges


In [ ]:
def match_rope(chain: Chain):
    e1, e2 = chain.edges
    if len(chain) > 2 and len(chain) % 2 == 1 and chain.is_hardend:
        if e1.dig == e2.dig:
            return e1.zone != e2.zone
        else:
            return e1.zone == e2.zone
    else:
        return False


def resolve_rope(board: Board, chain: Chain, scan_visible: Scanning) -> Resolving:
    """Cleanup all spoilers visible from both edges"""
    e1, e2 = chain.edges
    anchors = chain.anchors()
    spoilers = set(scan_visible(board, e1, e2)) - anchors
    if len(spoilers):
        yield Pattern(
            spoilers=spoilers,
            anchors={e1, e2},
            chain=chain,
        )

#### loop

Pattern: a closed alternating chain (odd number of links)

Rule: invalidate all visible from both edges of each soft link


In [ ]:
def match_loop(chain: Chain):
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def resolve_loop(board: Board, chain: Chain, scan_visible: Scanning) -> Resolving:
    """Cleanup all spoilers visible from both edges of each soft link"""
    anchors = chain.anchors()  # to exclude linking back to chain
    for link in filter(lambda lnk: isinstance(lnk, SLink), chain):
        e1, e2 = link
        spoilers = set(scan_visible(board, e1, e2)) - anchors
        if len(spoilers):
            yield Pattern(
                spoilers=spoilers,
                anchors={e1, e2},
                chain=chain,
            )

### Search for chains

- searching for all hard inks first
- trying to connect them into chains


In [ ]:
def expand_chain(current: Chain, links: Iterable[HLink], connect: Connecting) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""

    def close(chain):
        e1, e2 = chain.edges
        if len(chain) > 1 and chain.is_hardend:
            closing = connect(e2, e1)
            if closing:
                yield Chain.extend(chain, closing)

    def extend(chain, link):
        e1, e2 = chain.edges
        x1, x2 = link
        if conn := connect(e2, x1):
            yield Chain.extend(chain, conn, link)
        if conn := connect(e2, x2):
            yield Chain.extend(chain, conn, link.reversed())
        # if conn := connect(x2, e1):
        #     yield Chain.extendhead(chain, link, conn)
        # if conn := connect(x1, e1):
        #     yield Chain.extendhead(chain, link.reversed(), conn)

    anchors: set[Node] = current.anchors()

    def noncycling(lnk: Link):
        return lnk[0] not in anchors and lnk[1] not in anchors

    for link in filter(noncycling, links):
        for extended in extend(current, link):
            yield from close(extended)  # yield closed before open for breadth-first
            yield extended

In [ ]:
def search_chains(current: Board, links: Iterable[HLink], matching: Callable[[Chain], bool], connecting: Connecting, max_length: int = 8):
    counts = count_finals(current)

    def rate(link: Link):
        return min(counts[link[0].dig], counts[link[1].dig])

    links = sorted(links, key=rate)  # prioritize most present (least final-counted)
    init = [Chain((l,)) for l in links]

    def expanding(chain: Chain):
        yield from expand_chain(chain, links, connecting)

    def canceling(chain: Chain):
        return len(chain) >= max_length

    yield from search_breadth(init, expanding, matching, canceling)

In [ ]:
def chains_(types: str):

    def scanning(current: Board, n1: Node, n2: Node) -> set[Node]:
        visible = set()
        if "2" in types:
            visible |= set(scan_visible_2(current, n1, n2))
        if "1" in types or "3" in types:
            visible |= set(scan_visible_13(current, n1, n2))
        return visible

    def conecting(n1: Node, n2: Node) -> SLink | None:
        if "2" in types:
            return connect_soft_2(n1, n2) or connect_soft_13(n1, n2)
        elif "1" in types or "3" in types:
            return connect_soft_13(n1, n2)

    def matching(chain: Chain):
        return match_loop(chain) or match_rope(chain)

    def resolver(current: Board) -> Resolving:
        links = set()
        if "2" in types:
            links |= set(search_hard_2(current))
        if "3" in types:
            links |= set(search_hard_31(current))
        elif "1" in types:
            links |= set(search_hard_1(current))

        for chain in search_chains(current, links, matching=matching, connecting=conecting, max_length=6):
            if match_loop(chain):
                res = tuple(resolve_loop(current, chain, scanning))
            elif match_rope(chain):
                res = tuple(resolve_rope(current, chain, scanning))
            else:
                res = None

            if not res:
                continue  # if didn't work

            yield from res
            break  # on first worked

    resolver.__name__ = f"chains[{types}]"

    return resolver

In [ ]:
for _ in chains_("123")(puzzle):
    pprint(_)

# A puzzle


In [ ]:
from utils import fillempty, picture, parsepic, parsepic_wide

puzzle = parsepic("""
. . .│. 8 9│. . . 
. . .│. . .│1 7 . 
6 . .│. . .│. . . 
─────┼─────┼─────
. 2 .│3 . .│. . . 
. 1 .│. . .│. . 9 
. . .│. . .│. 6 8 
─────┼─────┼─────
8 . 9│. 5 .│. . . 
. . .│7 . .│2 . . 
5 . .│. . .│. . . 
""")

puzzle = Board.transform(puzzle, fillempty)

In [ ]:
# puzzle = await solve_silent(puzzle, open_singles, hidden_singles)

In [ ]:
puzzle = await solve_logging(
    puzzle,
    naked_singles,
    onceolver(hidden_singles),
    naked_multiples,
    onceolver(hidden_multiples),
    # onceolver(locked_triplets),
    # onceolver(chains_("123")),
    # randomchoice,
    filtout={"naked_singles", "hidden_singles"},
)

# GUI


In [ ]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-input-color: var(--vscode-editor-foreground);
    --jp-widgets-input-background-color: var(--vscode-editor-background);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.jupyter-widgets input {
   background-color: var(--jp-widgets-input-background-color);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [ ]:
import asyncio
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display

from traitlets import HasTraits, Instance, Set, Unicode, observe, Bool, Dict
from canvas import SudokuCanvas

In [ ]:
def click_future(button: w.Button) -> asyncio.Future[bool]:
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


# TODO: make it cancellable somehow

In [ ]:
class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Unicode()
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Node))
    empties = Set(Instance(Node))
    anchors = Set(Instance(Node))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()

        self._counters = {
            str(dig): w.Label(
                str(dig),
                layout=dict(width="auto", justify_content="center"),
                style=dict(text_color="black", background="var(--jp-info-color0)"),
            )
            for dig in DIGITS
        }
        self._counters["TOTAL"] = w.Label(
            "...",
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        self._status = w.Label(
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._status],
                    layout=dict(align_items="stretch", width="7em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self._hlayers = set()
        self.counters = count_finals(self.puzzle)

    def toggle_layer(self, layer: int):
        if not self.puzzle:
            return
        if layer in self._hlayers:
            self._hlayers.remove(layer)
        else:
            self._hlayers.add(layer)
        self._canvas.draw_board(self.puzzle, self._hlayers)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "var(--jp-info-color0)"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            w = self._counters[str(dig)]
            w.value = f"{dig}: ({cnt})"
        total = counters.total()
        w = self._counters["TOTAL"]
        w.value = f"Total: ({total})"

    @observe("targets", "anchors", "empties", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for lnk in self.links:
                self._highlight_link(lnk, "blue")

            for node in self.empties:
                self._highlight_node(node, "pink")

            for node in self.anchors:
                if node.is_cellular:
                    self._highlight_node(node, "cyan")
                else:
                    self._highlight_group(node, "cyan")

            for cell in self.targets:
                self._highlight_node(cell, "red")

    def reset_highlights(self):
        self._canvas.clear_highlights()
        self.targets = set()
        self.anchors = set()
        self.empties = set()
        self.links = set()

    def _highlight_node(self, node: Node, color: str):
        for loc in node.zone:
            for dig in node.digits:
                self._canvas.highlight_segment(loc, dig, color=color)

    def _highlight_cell(self, cell: Cell, color: str):
        for dig in cell.digits:
            self._canvas.highlight_segment(cell.loc, dig, color=color)

    def _highlight_link(self, lnk: Link, color: str):
        n1, n2 = lnk
        if n1.zone.is_cell and n2.zone.is_cell:
            self._canvas.highlight_link(
                n1.loc,
                n1.dig,
                n2.loc,
                n2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )
        else:
            n1locs = tuple(iter(n1.zone))
            l1mid = Loc(
                sum(l.r for l in n1locs) // len(n1locs),
                sum(l.c for l in n1locs) // len(n1locs),
            )
            n2locs = tuple(iter(n2.zone))
            l2mid = Loc(
                sum(l.r for l in n2locs) // len(n2locs),
                sum(l.c for l in n2locs) // len(n2locs),
            )
            self._canvas.highlight_link(
                l1mid,
                n1.dig,
                l2mid,
                n2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )

    def _highlight_group(self, node: Node, color: str):
        locs = tuple(iter(node.zone))
        lmin = Loc(min(l.r for l in locs), min(l.c for l in locs))
        lmax = Loc(max(l.r for l in locs), max(l.c for l in locs))
        self._canvas.highlight_link(lmin, node.dig, lmax, node.dig, style="GROUP", color=color)
        self._highlight_node(node, color)

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    def click_continue(self) -> asyncio.Future[bool]:
        return click_future(self._continue)

    async def pause(self):
        self.paused = True
        await self.click_continue()
        self.paused = False


# GUI meta-widget


LINK_STYLES = {
    "Link": "SOLID",
    "HLink": "HARD",
    "SLink": "SOFT",
}

In [ ]:
debug_view = w.Output()
gui = GUI()

In [ ]:
display(gui, debug_view)

In [ ]:
gui.puzzle = puzzle
gui.targets = set()
gui.anchors = set()
gui.links = set()
gui.status = "..."

In [ ]:
# links = search_hard_31(puzzle)
chains = chains_("3")(puzzle)

In [ ]:
gui.links = set()
gui.anchors = set()
# link = next(links)
pattern = next(chains)
chain = pattern["chain"]
print(pattern)
gui.links = set(chain)
gui.anchors = set(chain.edges)

In [ ]:
gui.links

In [ ]:
async def solve_ui(initial: Board, *resolvers: Resolver, filtout: set[str] = set()):
    current = initial
    result = initial

    _, _, drafted = result.validate()
    assert drafted

    gui.puzzle = current
    gui.running = True
    gui.inspecting = {r.__name__: r.__name__ not in filtout for r in resolvers}

    try:
        iteration = 0
        async for resolver, pattern, result in solver(initial, *resolvers):
            iteration += 1
            # print(iteration, resolver.__name__)
            # pprint(pattern)
            if gui.inspecting[resolver.__name__]:
                gui.puzzle = current
                resolving = f"#{iteration} {resolver.__name__}: ..."
                gui.resolving = resolving
                render_resolution(pattern)
                await gui.pause()
                clear_resolution()
                gui.puzzle = result
                await asyncio.sleep(0.2)
            current = result
            gui.resolving = f"#{iteration}"

            complete, valid, drafted = result.validate()
            render_status(complete, valid, drafted)
            if complete or not valid or not drafted:
                break
    except Exception as e:
        # FIXME: the cancel button handler
        with debug_view:
            raise RuntimeError("Solver failed") from e

    complete, valid, drafted = result.validate()
    render_status(complete, valid, drafted)
    gui.resolving = f"#{iteration} ENDED"
    gui.puzzle = result
    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Pattern):
    with hold_canvas():
        gui.targets = res.get("spoilers", set())
        gui.anchors = res.get("anchors", set())
        gui.empties = res.get("space", set())
        if "chain" in res:
            gui.links = set(res["chain"])
        elif "links" in res:
            gui.links = res["links"]
        else:
            gui.links = set()


def render_status(complete: bool, valid: bool, drafted: bool):
    if not valid:
        gui.status = "BROKEN"
    elif complete:
        gui.status = "SOLVED"
    elif not drafted:
        gui.status = "STUCK"
    else:
        gui.status = "..."


def clear_resolution():
    gui.reset_highlights()

In [ ]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        naked_singles,
        onceolver(hidden_singles),
        onceolver(naked_multiples),
        onceolver(hidden_multiples),
        # onceolver(locked_triplets),
        onceolver(chains_("13")),
        # randomchoice,
        filtout={"naked_singles", "hidden_singles"},
    )
)

In [ ]:
task

In [ ]:
task.cancel()  # it breaks something